# 03 - LangChain intermediário

Próximo passo depois da chamada simples. Neste notebook usamos:

1. **Templates de prompt reutilizáveis** (`ChatPromptTemplate`)
2. **Composição com cadeias** (LCEL, operador `|`)
3. **Saída estruturada** (`with_structured_output` com Pydantic)
4. **Parsers** (`StrOutputParser` e `PydanticOutputParser`)

As configurações de chave e modelo continuam vindo do `.env` na raiz do projeto.

In [1]:
import os

from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from langchain_openai import ChatOpenAI

load_dotenv(find_dotenv())

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-5-mini"),
    api_key=os.getenv("OPENAI_API_KEY"),
)

## 1. Templates de prompt reutilizáveis

O template separa a instrução fixa das variáveis. Assim o mesmo prompt serve para vários assuntos.

In [2]:
from langchain_core.prompts import ChatPromptTemplate

template_explicacao = ChatPromptTemplate.from_messages(
    [
        ("system", "Você é um professor de {materia} que explica conceitos de forma clara e objetiva."),
        ("human", "Explique em até {limite_palavras} palavras: {conceito}"),
    ]
)

template_explicacao.format_messages(materia="física", limite_palavras=30, conceito="gravitação")

[SystemMessage(content='Você é um professor de física que explica conceitos de forma clara e objetiva.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Explique em até 30 palavras: gravitação', additional_kwargs={}, response_metadata={})]

## 2. Composição com cadeias (LCEL)

Com `|` encadeamos prompt, modelo e parser. Cada etapa recebe a saída da anterior.

In [3]:
from langchain_core.output_parsers import StrOutputParser

cadeia_explicacao = template_explicacao | llm | StrOutputParser()

explicacao = cadeia_explicacao.invoke(
    {"materia": "física", "conceito": "gravitação universal", "limite_palavras": 60}
)

display(Markdown(f"**Explicação:**\n\n{explicacao}"))

**Explicação:**

A gravitação universal, formulada por Newton, afirma que toda massa atrai outra com força F = G m1 m2/r^2 (G é a constante gravitacional), proporcional ao produto das massas e inversamente proporcional ao quadrado da distância entre seus centros; a força age ao longo da linha que as une.

Com `batch` a mesma cadeia roda para várias entradas de uma vez.

In [4]:
conceitos = ["inércia", "entropia", "fotossíntese"]
entradas = [{"materia": "ciência", "conceito": c, "limite_palavras": 25} for c in conceitos]

resultados = cadeia_explicacao.batch(entradas)

for conceito, texto in zip(conceitos, resultados):
    display(Markdown(f"**{conceito.capitalize()}:** {texto}"))

**Inércia:** Inércia é a resistência de um corpo à mudança de seu estado de movimento; corpos em repouso ou em movimento tendem a manter esse estado.

**Entropia:** Entropia é a medida da desordem ou do número de microestados; quantifica a energia indisponível para trabalho e tende a aumentar.

**Fotossíntese:** Fotossíntese é o processo pelo qual plantas, algas e algumas bactérias convertem luz, água e CO2 em glicose e oxigênio.

Agora encadeamos duas cadeias: a explicação gerada vira a entrada de uma cadeia de tradução. `RunnableLambda` adapta o formato entre as duas.

In [5]:
from langchain_core.runnables import RunnableLambda

template_traducao = ChatPromptTemplate.from_messages(
    [
        ("system", "Você traduz textos para {idioma} mantendo o sentido original."),
        ("human", "{texto}"),
    ]
)

cadeia_traducao = template_traducao | llm | StrOutputParser()

pipeline = (
    cadeia_explicacao
    | RunnableLambda(lambda texto: {"idioma": "inglês", "texto": texto})
    | cadeia_traducao
)

traducao = pipeline.invoke(
    {"materia": "biologia", "conceito": "fotossíntese", "limite_palavras": 40}
)

display(Markdown(f"**Tradução:**\n\n{traducao}"))

**Tradução:**

Photosynthesis is the process by which plants, algae, and cyanobacteria use light, water, and carbon dioxide to produce glucose and release oxygen, occurring in chloroplasts via chlorophyll.

## 3. Saída estruturada com Pydantic

Definimos o formato esperado como uma classe. `with_structured_output` faz o modelo devolver um objeto desse tipo, já validado.

In [6]:
from typing import List

from pydantic import BaseModel, Field


class Cientista(BaseModel):
    nome: str = Field(description="Nome completo")
    nacionalidade: str
    ano_nascimento: int
    areas: List[str] = Field(description="Áreas de atuação")
    principal_contribuicao: str


template_perfil = ChatPromptTemplate.from_messages(
    [
        ("system", "Você extrai informações biográficas precisas e responde em português."),
        ("human", "Descreva o perfil de {pessoa}."),
    ]
)

cadeia_perfil = template_perfil | llm.with_structured_output(Cientista)

perfil = cadeia_perfil.invoke({"pessoa": "Marie Curie"})

display(
    Markdown(
        f"**Nome:** {perfil.nome}  \n"
        f"**Nacionalidade:** {perfil.nacionalidade}  \n"
        f"**Nascimento:** {perfil.ano_nascimento}  \n"
        f"**Áreas:** {', '.join(perfil.areas)}  \n"
        f"**Principal contribuição:** {perfil.principal_contribuicao}"
    )
)

**Nome:** Marie Skłodowska-Curie  
**Nacionalidade:** Polonesa (naturalizada francesa)  
**Nascimento:** 1867  
**Áreas:** Física, Química, Radioatividade, Química radiológica  
**Principal contribuição:** Pesquisa pioneira sobre radioatividade; descoberta dos elementos polônio e rádio; desenvolvimento de técnicas para isolar elementos radioativos; primeira pessoa a receber dois Prêmios Nobel em áreas diferentes (Física, 1903; Química, 1911).

## 4. Parser Pydantic (alternativa sem saída nativa)

Quando o modelo ou provedor não suporta saída estruturada, o `PydanticOutputParser` gera instruções de formato para o prompt e interpreta o texto devolvido.

In [7]:
from langchain_core.output_parsers import PydanticOutputParser

parser = PydanticOutputParser(pydantic_object=Cientista)

template_parser = ChatPromptTemplate.from_messages(
    [
        ("system", "Responda somente no formato pedido.\n{format_instructions}"),
        ("human", "Descreva o perfil de {pessoa}."),
    ]
).partial(format_instructions=parser.get_format_instructions())

cadeia_parser = template_parser | llm | parser

perfil_newton = cadeia_parser.invoke({"pessoa": "Isaac Newton"})

display(Markdown(f"**{perfil_newton.nome}** ({perfil_newton.ano_nascimento}): {perfil_newton.principal_contribuicao}"))

**Isaac Newton** (1643): Formulação das leis do movimento e da lei da gravitação universal; desenvolvimento do cálculo diferencial e integral; avanços fundamentais na óptica.